# 🔍 Notebook 5: Error Analysis

**Hindi-specific error analysis for CRF and mBERT**

This notebook:
1. Categorizes errors by type (boundary, type confusion, missed, spurious)
2. Analyzes Hindi-specific error patterns
3. Compares error distributions between CRF and mBERT
4. Selects illustrative examples for the report
5. Investigates script-related errors (Devanagari vs Latin)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.error_analysis import (
    analyze_all_errors,
    error_statistics,
    print_error_summary,
    get_interesting_examples,
    format_error_example,
    save_error_analysis,
    ErrorType,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load Predictions

In [ ]:
with open('../results/predictions/crf_predictions.json', 'r', encoding='utf-8') as f:
    crf_predictions = json.load(f)

with open('../results/predictions/mbert_predictions.json', 'r', encoding='utf-8') as f:
    mbert_predictions = json.load(f)

print(f"CRF predictions:   {len(crf_predictions)} sentences")
print(f"mBERT predictions: {len(mbert_predictions)} sentences")

## 2. Analyze CRF Errors

In [ ]:
# Find all CRF errors
crf_errors = analyze_all_errors(crf_predictions)
crf_stats = error_statistics(crf_errors)
print_error_summary(crf_stats, 'CRF')

## 3. Analyze mBERT Errors

In [ ]:
# Find all mBERT errors
mbert_errors = analyze_all_errors(mbert_predictions)
mbert_stats = error_statistics(mbert_errors)
print_error_summary(mbert_stats, 'mBERT')

## 4. Compare Error Distributions

In [ ]:
# Side-by-side error type comparison
all_error_types = sorted(set(
    list(crf_stats['error_type_counts'].keys()) + 
    list(mbert_stats['error_type_counts'].keys())
))

comparison_data = []
for et in all_error_types:
    comparison_data.append({
        'Error Type': et,
        'CRF': crf_stats['error_type_counts'].get(et, 0),
        'mBERT': mbert_stats['error_type_counts'].get(et, 0),
    })

error_df = pd.DataFrame(comparison_data)
print("\nError Type Comparison:")
print(error_df.to_string(index=False))

In [ ]:
# Plot error comparison
import numpy as np

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(all_error_types))
width = 0.35

crf_counts = [crf_stats['error_type_counts'].get(et, 0) for et in all_error_types]
mbert_counts = [mbert_stats['error_type_counts'].get(et, 0) for et in all_error_types]

ax.bar(x - width/2, crf_counts, width, label='CRF', color='#4C72B0', alpha=0.85)
ax.bar(x + width/2, mbert_counts, width, label='mBERT', color='#DD8452', alpha=0.85)

ax.set_xlabel('Error Type')
ax.set_ylabel('Count')
ax.set_title('Error Distribution: CRF vs mBERT', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(all_error_types, rotation=30, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/error_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Interesting Error Examples

**Select 2–4 of these for your report and presentation.**

In [ ]:
# CRF interesting examples
print("=" * 70)
print("INTERESTING CRF ERROR EXAMPLES")
print("=" * 70)

crf_examples = get_interesting_examples(crf_errors, n=8)
for i, ex in enumerate(crf_examples):
    print(f"\n--- CRF Error Example {i+1} ---")
    print(format_error_example(ex))
    print()

In [ ]:
# mBERT interesting examples
print("=" * 70)
print("INTERESTING mBERT ERROR EXAMPLES")
print("=" * 70)

mbert_examples = get_interesting_examples(mbert_errors, n=8)
for i, ex in enumerate(mbert_examples):
    print(f"\n--- mBERT Error Example {i+1} ---")
    print(format_error_example(ex))
    print()

## 6. Key Findings Summary

Summarize the error analysis for your report.

In [ ]:
print("\n" + "=" * 70)
print("KEY FINDINGS — ERROR ANALYSIS")
print("=" * 70)

print(f"\n1. Total errors:")
print(f"   CRF:   {crf_stats['total_errors']}")
print(f"   mBERT: {mbert_stats['total_errors']}")

# Which error type improved most?
print(f"\n2. Error type improvements (CRF → mBERT):")
for et in all_error_types:
    crf_count = crf_stats['error_type_counts'].get(et, 0)
    mbert_count = mbert_stats['error_type_counts'].get(et, 0)
    if crf_count > 0:
        reduction = (crf_count - mbert_count) / crf_count * 100
        direction = '↓' if reduction > 0 else '↑'
        print(f"   {et:30s}: {crf_count:5d} → {mbert_count:5d} ({direction} {abs(reduction):.1f}%)")

print(f"\n3. Script-related errors:")
print(f"   CRF:   {crf_stats['script_distribution']}")
print(f"   mBERT: {mbert_stats['script_distribution']}")

## 7. Save Error Analysis

In [ ]:
# Save error analysis
save_error_analysis(crf_errors, crf_stats, '../results/crf_error_analysis.json')
save_error_analysis(mbert_errors, mbert_stats, '../results/mbert_error_analysis.json')

# Save comparison table
error_df.to_csv('../results/tables/error_comparison.csv', index=False)
print("\n✓ All error analysis saved")